In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [ ]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### Fix problem

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl
wanted_issues = [
    "New",
]

df = df[
    df["ISSUE"].str.strip().isin(wanted_issues) &
    (df["Network ID"] != 11)
]

len(df)

df =  df[['ISSUE','Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)

df["Network ID"] = pd.to_numeric(df["Network ID"], errors="coerce").astype("Int64")
df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")


df['Station_name'] = df['Unique Names'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df = df.drop(columns=['Unique Names'])

df['native_id'] = df['Native ID'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df = df.drop(columns=['Native ID'])


import re

def parse_lat_lon_elev(text):
    # regex to capture: lon value, W/E, lat value, N/S, elevation
    pattern = r'([\d\.]+)\s*([WE]),\s*([\d\.]+)\s*([NS]),\s*Elev\.\s*([\d\.]+)'
    m = re.search(pattern, text)
    
    if not m:
        return None, None, None

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    # Convert numbers
    lon = float(lon_val) * (-1 if lon_dir == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir == "S" else 1)
    elev = float(elev_val)

    return lat, lon, elev

# Apply to dataframe
df[['lat', 'lon', 'Elevation']] = df['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_lat_lon_elev(x))
)
df = df.drop(columns=['Unique Location (String)'])

df["Network Name"] = df["Network Name"].str.replace(r'.*->\s*', '', regex=True)

df


,ISSUE,Station ID,Network ID,Network Name,Station_name,native_id,lat,lon,Elevation
0,New,<NA>,<NA>,BCH-SITEC,Attachie Flat Lower Terrace,AFL,NaN,NaN,NaN
1,New,<NA>,<NA>,BCH-SITEC,Attachie Plateau,ATP,56.233,-121.466,646.0
2,New,<NA>,<NA>,BCH-SITEC,Bear Flat,BEF,56.120,-121.213,475.0
3,New,<NA>,<NA>,BCH-SITEC,Farrell Creek,FLC,56.120,-121.701,472.0
4,New,<NA>,<NA>,BCH-SITEC,Taylor Bison,TAB,56.167,-120.758,411.0
5,New,<NA>,<NA>,BCH-SITEC,Tea Creek,TEC,56.237,-120.954,653.0


### The Network Name (BCH-SITEC) need to be update first

In [5]:
from sqlalchemy import text

SQL_GET_NETWORK_ID = text("""
SELECT network_id
FROM meta_network
WHERE LOWER(network_name) = LOWER(:network_name)
""")

def get_network_id(network_name, engine):
    """Return the network_id for a given network_name."""
    with engine.connect() as conn:
        result = conn.execute(SQL_GET_NETWORK_ID, {"network_name": network_name}).scalar()
    return result

# Make sure 'Network Name' column exists
df['Network ID'] = df['Network Name'].apply(lambda name: get_network_id(name, engine))
df

,ISSUE,Station ID,Network ID,Network Name,Station_name,native_id,lat,lon,Elevation
0,New,<NA>,70,BCH-SITEC,Attachie Flat Lower Terrace,AFL,NaN,NaN,NaN
1,New,<NA>,70,BCH-SITEC,Attachie Plateau,ATP,56.233,-121.466,646.0
2,New,<NA>,70,BCH-SITEC,Bear Flat,BEF,56.120,-121.213,475.0
3,New,<NA>,70,BCH-SITEC,Farrell Creek,FLC,56.120,-121.701,472.0
4,New,<NA>,70,BCH-SITEC,Taylor Bison,TAB,56.167,-120.758,411.0
5,New,<NA>,70,BCH-SITEC,Tea Creek,TEC,56.237,-120.954,653.0


In [6]:
from sqlalchemy import func

stations_created = []

# for _, row in df.iloc[0:2].iterrows():
for _, row in df.iterrows():
    
    name = row['Station_name']
    nid  = row['native_id']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=row['Network ID'])


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['Elevation'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('Attachie Flat Lower Terrace', 14927), ('Attachie Plateau', 14928), ('Bear Flat', 14929), ('Farrell Creek', 14930), ('Taylor Bison', 14931), ('Tea Creek', 14932)]
